In [1]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate
from langchain_core.documents import Document
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains.retrieval import create_retrieval_chain
import os 
from dotenv import load_dotenv
load_dotenv()

e:\RAG AGENTIC AI\RAG LEARNING\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
python-dotenv could not parse statement starting at line 7


True

In [2]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableMap

In [3]:
# step 1: Load and split the dataset

loader = TextLoader("langchain_crewai_dataset.txt")
row_docs = loader.load()
splitter = RecursiveCharacterTextSplitter(chunk_size = 300, chunk_overlap =50)
chunks = splitter.split_documents(row_docs)

In [4]:
# step 2 : Vector store
embedding_model = HuggingFaceEmbeddings(model="all-MiniLM-L6-v2")


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 309.44it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [5]:
vectorstore = FAISS.from_documents(chunks,embedding_model)

# step 3 : MMR Retriver
retriver = vectorstore.as_retriever(search_type ="mmr", search_kwargs={"k":5})


In [6]:
# step 4: LLM and Prompt

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

In [7]:
llm=init_chat_model(
    model="llama-3.1-8b-instant",
    model_provider="groq"
)

In [8]:
# Step 3: Query decomposition
decomposition_prompt = PromptTemplate.from_template("""
You are an AI assistant. Decompose the following complex question into 2 to 4 smaller sub-questions for better document retrieval.

Question: "{question}"

Sub-questions:
""")
decomposition_chain = decomposition_prompt | llm | StrOutputParser()

In [9]:
query = "How does LangChain use memory and agents compared to CrewAI?"
decomposition_question=decomposition_chain.invoke({"question": query})


In [10]:
print(decomposition_question)

To better understand the comparison between LangChain and CrewAI, I'll break down the complex question into smaller sub-questions:

1. **What is LangChain's memory architecture?** 
    - This sub-question will help us understand how LangChain stores and retrieves information, which is crucial for comparing it to CrewAI.

2. **How do agents interact with memory in LangChain?**
    - This sub-question will provide insights into the agent-memory interaction in LangChain, enabling us to compare it to CrewAI's agent-memory interaction.

3. **What is CrewAI's approach to memory and agent interactions?**
    - This sub-question will serve as a comparison baseline, allowing us to understand CrewAI's architecture and how it differs from LangChain's.

4. (Optional) **What are the key benefits or trade-offs of LangChain's memory and agent architecture compared to CrewAI?**
    - This sub-question will help us identify the advantages or disadvantages of each system, providing a more comprehensive 

In [11]:
# Step 4: QA chain per sub-question
qa_prompt = PromptTemplate.from_template("""
Use the context below to answer the question.

Context:
{context}

Question: {input}
""")
qa_chain = create_stuff_documents_chain(llm=llm, prompt=qa_prompt)

In [12]:
# Step 5: Full RAG pipeline logic
def full_query_decomposition_rag_pipeline(user_query):
    # Decompose the query
    sub_qs_text = decomposition_chain.invoke({"question": user_query})
    sub_questions = [q.strip("-•1234567890. ").strip() for q in sub_qs_text.split("\n") if q.strip()]
    
    results = []
    for subq in sub_questions:
        docs = retriver.invoke(subq)
        result = qa_chain.invoke({"input": subq, "context": docs})
        results.append(f"Q: {subq}\nA: {result}")
    
    return "\n\n".join(results)

In [13]:
# Step 6: Run
query = "How does LangChain use memory and agents compared to CrewAI?"
final_answer = full_query_decomposition_rag_pipeline(query)
print("✅ Final Answer:\n")
print(final_answer)

✅ Final Answer:

Q: To better facilitate document retrieval and provide more accurate information, I can decompose the complex question into the following sub-questions:
A: It appears that the context you provided is about the LangChain system. Given the description of its capabilities in retrieving information from large document corpora, it seems that you would decompose the complex question into sub-questions like this:

- How can I utilize LangChain's semantic search functionality to retrieve relevant documents from a vector database like FAISS, Chroma, Pinecone, or Weaviate?
- What are the standard patterns like Stuff, Map-Reduce, and Refine that I can use to compose and reuse chains within LangChain?
- How can I integrate external knowledge from a vector database into a Language Model (LLM) using Retrieval-Augmented Generation (RAG)?

Q: "What is the role of memory in LangChain?"
A: Based on the provided context, the role of memory in LangChain is to allow the LLM to:

1. Maintai